In [11]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
BUCKET_NAME = 'spacenet-dataset'
# On s'arrête au dossier de la ville pour voir ce qu'il contient
GERMANY_PREFIX = 'spacenet/SN8_floods/Germany_Training_Public/'

print(f"Exploration de {GERMANY_PREFIX}...")
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=GERMANY_PREFIX, Delimiter='/')

if 'CommonPrefixes' in response:
    print("\n--- SOUS-DOSSIERS TROUVÉS ---")
    for prefix in response['CommonPrefixes']:
        print(prefix.get('Prefix'))
else:
    print("Toujours rien ? On tente une recherche de fichiers directe...")
    # Si pas de dossiers, on cherche les 5 premiers fichiers n'importe où dans l'arborescence
    files = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=GERMANY_PREFIX, MaxKeys=5)
    for obj in files.get('Contents', []):
        print(f"Fichier trouvé : {obj['Key']}")

Exploration de spacenet/SN8_floods/Germany_Training_Public/...

--- SOUS-DOSSIERS TROUVÉS ---
spacenet/SN8_floods/Germany_Training_Public/POST-event/
spacenet/SN8_floods/Germany_Training_Public/PRE-event/
spacenet/SN8_floods/Germany_Training_Public/annotations/


In [12]:
# Chemins vers les dossiers que nous venons de découvrir
POST_EVENT_PREFIX = 'spacenet/SN8_floods/Germany_Training_Public/POST-event/'
ANNOTATIONS_PREFIX = 'spacenet/SN8_floods/Germany_Training_Public/annotations/'

print("--- Listing des images POST-event ---")
post_images = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=POST_EVENT_PREFIX, MaxKeys=5)
for obj in post_images.get('Contents', []):
    print(f"IMAGE : {obj['Key']}")

print("\n--- Listing des annotations (Masques) ---")
annotations = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=ANNOTATIONS_PREFIX, MaxKeys=5)
for obj in annotations.get('Contents', []):
    print(f"MASQUE : {obj['Key']}")

--- Listing des images POST-event ---
IMAGE : spacenet/SN8_floods/Germany_Training_Public/POST-event/1040050035DC3B00_0_15_63.tif
IMAGE : spacenet/SN8_floods/Germany_Training_Public/POST-event/1040050035DC3B00_0_15_65.tif
IMAGE : spacenet/SN8_floods/Germany_Training_Public/POST-event/1040050035DC3B00_0_15_66.tif
IMAGE : spacenet/SN8_floods/Germany_Training_Public/POST-event/1040050035DC3B00_0_15_67.tif
IMAGE : spacenet/SN8_floods/Germany_Training_Public/POST-event/1040050035DC3B00_0_15_68.tif

--- Listing des annotations (Masques) ---
MASQUE : spacenet/SN8_floods/Germany_Training_Public/annotations/0_15_63.geojson
MASQUE : spacenet/SN8_floods/Germany_Training_Public/annotations/0_15_65.geojson
MASQUE : spacenet/SN8_floods/Germany_Training_Public/annotations/0_15_66.geojson
MASQUE : spacenet/SN8_floods/Germany_Training_Public/annotations/0_15_67.geojson
MASQUE : spacenet/SN8_floods/Germany_Training_Public/annotations/0_15_68.geojson


In [13]:
import os

# 1. Définition des constantes de chemins (Project Structure)
RAW_DATA_PATH = '../data/raw/'
PROCESSED_PATH = '../data/processed/'
MODELS_PATH = '../models/'

# 2. Création automatique des dossiers s'ils n'existent pas
# Cela évite les erreurs de type "FileNotFoundError" plus tard
os.makedirs(RAW_DATA_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

print(f"✅ Chemins configurés ! Les données brutes iront dans : {RAW_DATA_PATH}")

✅ Chemins configurés ! Les données brutes iront dans : ../data/raw/


In [14]:
import os

# 1. Définition des clés exactes trouvées au listing précédent
# On choisit l'échantillon 0_15_63
REMOTE_IMAGE_KEY = 'spacenet/SN8_floods/Germany_Training_Public/POST-event/1040050035DC3B00_0_15_63.tif'
REMOTE_MASK_KEY = 'spacenet/SN8_floods/Germany_Training_Public/annotations/0_15_63.geojson'

# 2. Définition des noms de fichiers locaux
local_image_path = os.path.join(RAW_DATA_PATH, 'sample_post_event_0_15_63.tif')
local_mask_path = os.path.join(RAW_DATA_PATH, 'sample_mask_0_15_63.geojson')

# 3. Téléchargement
print(f"Téléchargement de l'image satellite...")
s3.download_file(BUCKET_NAME, REMOTE_IMAGE_KEY, local_image_path)

print(f"Téléchargement du masque (GeoJSON)...")
s3.download_file(BUCKET_NAME, REMOTE_MASK_KEY, local_mask_path)

print(f"\n--- SUCCÈS ---")
print(f"Image stockée : {local_image_path} ({os.path.getsize(local_image_path) / 1024:.2f} KB)")
print(f"Masque stocké : {local_mask_path} ({os.path.getsize(local_mask_path) / 1024:.2f} KB)")

Téléchargement de l'image satellite...
Téléchargement du masque (GeoJSON)...

--- SUCCÈS ---
Image stockée : ../data/raw/sample_post_event_0_15_63.tif (8906.00 KB)
Masque stocké : ../data/raw/sample_mask_0_15_63.geojson (10.62 KB)
